# Dumb autoregressive generation from an MLM encoder

This model was trained with **masked language modelling** — bidirectional, no causal mask, no
next-token objective. It is not a generative model. But we can *coerce* it into generating:

1. take a prompt,
2. append **3 `[MASK]` tokens**,
3. read the prediction at the **first** mask slot, commit that token,
4. repeat.

The 3-mask lookahead matters. A mask sitting at the very end of a sequence is a distribution the
model rarely saw in training (masks were sprinkled *inside* packed text), so giving it a couple of
trailing masks makes the position look more like training data — "more text follows here".

**Why this is legitimately dumb:** an autoregressive LM factorises `p(x) = Π p(x_t | x_<t)`, and
each step is trained to be exactly that conditional. Here every prediction is
`p(x_t | bidirectional context, with masks as placeholders)` — the model was never trained to
extend text, and the 3 mask slots are predicted *conditionally independently* of each other. We
throw away slots 2 and 3 precisely because they don't account for what slot 1 will become.

There is **no end-of-sentence token** in the vocabulary we can rely on (`<|endoftext|>` = 50279 is
a *document* separator used to pack the corpus), so generation stops on a token budget, on that
separator, or on sentence-final punctuation.

In [35]:
import os, sys, torch
sys.path.insert(0, ".")             # notebooks are run from the repo root
from mini_enc_transformer import build_tokenizer, build_model

# Must match the pretraining config exactly, or the state dict will not load.
class C:
    d_model, d_k, d_v, n_heads, n_layers, d_embed = 768, 64, 64, 4, 4, 128

SEQ_LEN  = 128     # the context length the model was TRAINED on -- the sliding window matches it
N_MASKS  = 3       # trailing [MASK] tokens; we only ever commit the first

CKPT = next(p for p in ["./checkpoints/ckpt3/last.pt", "./checkpoints/ckpt2/last.pt", "./checkpoints/ckpt/last.pt"]
            if os.path.exists(p))
print("checkpoint:", CKPT)

checkpoint: ./ckpt3/last.pt


In [37]:
CKPT = './checkpoints/ckpt2/last.pt'

In [38]:
tk, ids = build_tokenizer("allenai/OLMo-1B-hf")
model = build_model(C(), ids)
model.load_state_dict(torch.load(CKPT, map_location="cpu")["model"])
model.eval()

MASK_ID, EOS_ID = ids["mask_id"], tk.eos_token_id
print(f"vocab {ids['vocab_size']} | mask_id {MASK_ID} | eos/doc-sep {EOS_ID}")
print(f"params {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

vocab 50281 | mask_id 50280 | eos/doc-sep 50279
params 28.7M


In [27]:
@torch.no_grad()
def generate(prompt, max_gen=300, temperature=0.0, top_k=None, stop_on={".", "!", "?"}, verbose=False):
    """Mask-fill decoding: append N_MASKS masks, commit the first slot, repeat.

    add_special_tokens=False is essential -- this tokenizer appends <|endoftext|> to the
    prompt otherwise, the model reads it as a document boundary, and generation starts an
    unrelated new document instead of continuing the sentence.
    """
    toks = tk(prompt, add_special_tokens=False)["input_ids"] # tokenize prompt, without adding <|endoftext|>

    for _ in range(max_gen):
        ctx = toks[-(SEQ_LEN - N_MASKS):]                 # sliding window, matches training length
        x = torch.tensor([ctx + [MASK_ID] * N_MASKS])
        logits = model(x)["logits"][0, len(ctx)]          # FIRST mask slot only
        if temperature <= 0:
            nxt = int(logits.argmax())
        else:
            z = logits / temperature
            if top_k:
                z[z < torch.topk(z, top_k).values[-1]] = -float("inf")
            nxt = int(torch.multinomial(torch.softmax(z, -1), 1))
        if nxt == EOS_ID:
            break
        toks.append(nxt)
        if verbose:
            print(repr(tk.decode([nxt])), end=" ")
        if tk.decode([nxt]).strip() in stop_on:
            break
    return tk.decode(toks)

In [55]:
PROMPTS = [
    "Once upon a time there was a little girl who",
    "The movie was",
    "The capital of France is",
    "I watched this film last night and",
    "I watched X-Men: Days. The name of the movie was ",
    "Goku"
]
for p in PROMPTS:
    print(f"> {p}\n  {generate(p, max_gen=20)}\n")

> Once upon a time there was a little girl who
  Once upon a time there was a little girl who loved to play with her friends.

> The movie was
  The movie was made on the TV.

> The capital of France is
  The capital of France is the capital of France.

> I watched this film last night and
  I watched this film last night and was so excited to see the film.

> I watched X-Men: Days. The name of the movie was 
  I watched X-Men: Days. The name of the movie was icky.

> Goku
  Goku.



In [57]:
# Greedy decoding falls into loops ("a bit more than ... a bit more than"). Sampling helps,
# at the cost of coherence -- the usual trade, exaggerated here because the model has no
# next-token objective holding the sequence together.
torch.manual_seed(0)
for p in PROMPTS:
    print(f"> {p}")
    for t in (0.7, 1.0):
        print(f"  T={t}: {generate(p, temperature=t, top_k=40, max_gen=20)}")
    print()

> Once upon a time there was a little girl who
  T=0.7: Once upon a time there was a little girl who loved feeling sad for being sad and wanted to leave.
  T=1.0: Once upon a time there was a little girl who was excited to go to the park.

> The movie was
  T=0.7: The movie was made out of the film.
  T=1.0: The movie was so great, just just like a man, just like something else to see.

> The capital of France is
  T=0.7: The capital of France is a country who is not a country who is not a country who is not a country who is not
  T=1.0: The capital of France is a capital whose capital of France is another capital whose capital of France is a capital whose capital of France

> I watched this film last night and
  T=0.7: I watched this film last night and the man and the man and the man and his other family were so excited!
  T=1.0: I watched this film last night and, with the kids being over to all that amazing show ever.

> I watched X-Men: Days. The name of the movie was 
  T=0.7: I 

In [58]:
# Why we discard slots 2 and 3: all three masks are predicted from the SAME encoder pass and
# are conditionally independent of one another. Slot 2 does not know what slot 1 became, so
# reading them off together gives an incoherent phrase -- this is the core reason an MLM is
# not a generator.
@torch.no_grad()
def peek_all_masks(prompt):
    toks = tk(prompt, add_special_tokens=False)["input_ids"]
    x = torch.tensor([toks + [MASK_ID] * N_MASKS])
    lg = model(x)["logits"][0]
    picks = [int(lg[len(toks) + i].argmax()) for i in range(N_MASKS)]
    print(f"> {prompt}")
    print(f"   all 3 slots at once : {prompt}{tk.decode(picks)!r}")
    print(f"   committing 1 at a time: {generate(prompt, max_gen=N_MASKS, stop_on={})!r}\n")

for p in PROMPTS:
    peek_all_masks(p)

> Once upon a time there was a little girl who
   all 3 slots at once : Once upon a time there was a little girl who' loved to to'
   committing 1 at a time: 'Once upon a time there was a little girl who loved to play'

> The movie was
   all 3 slots at once : The movie was' made..'
   committing 1 at a time: 'The movie was made on the'

> The capital of France is
   all 3 slots at once : The capital of France is' the..'
   committing 1 at a time: 'The capital of France is the capital of'

> I watched this film last night and
   all 3 slots at once : I watched this film last night and' was was.'
   committing 1 at a time: 'I watched this film last night and was so excited'

> I watched X-Men: Days. The name of the movie was 
   all 3 slots at once : I watched X-Men: Days. The name of the movie was 'icky..'
   committing 1 at a time: 'I watched X-Men: Days. The name of the movie was icky.'

> Goku
   all 3 slots at once : Goku'...'
   committing 1 at a time: 'Goku.'



## What to expect, and what it does and doesn't tell you

- **Fluent-but-looping output is the norm.** Greedy decoding on an MLM collapses into repetition
  faster than on a causal LM, because nothing in training ever rewarded advancing a sequence.
- **Story prompts do best.** After phase 3 the mixture is ~39% TinyStories, so
  "Once upon a time..." continues far more sensibly than a factual prompt. That is the training
  distribution showing through, not reasoning.
- **Do not read this as a capability benchmark.** It is a qualitative probe of what the encoder
  has absorbed. The quantitative measures that matter are MLM accuracy on held-out text and the
  SST-2 fine-tune.
- The checkpoint auto-selects `ckpt3` → `ckpt2` → `ckpt`. If `ckpt3` is mid-training, you are
  sampling a partially-trained model; set `CKPT` explicitly to compare phases.

---

# Fixing the looping: a mild WSD fine-tune on CPU

The loops above are **not** mainly an EOS problem. They are an *objective mismatch*: the model was
trained to **infill** masks inside packed text, but we are asking it to **continue** text. Those are
different conditionals, and nothing in pretraining ever rewarded advancing a sequence.

So we train on the exact shape used at inference:

```
input  = [ ...left context... ] [MASK] [MASK] [MASK]
labels =                          next_tok  ignore  ignore
```

Only **slot 1** is supervised — that is next-token prediction expressed through the mask interface.
Two deliberate choices on top:

1. **EOS oversampling (30%)** — the narrower fix. `<|endoftext|>` separates documents and is rarely
   a masked target in pretraining, so the model never learned where text ends. We sample 30% of
   examples at real document boundaries.
2. **Randomised context length (8–125)** — at inference the window *grows*: the first generated
   token sees only the prompt, later ones see the full 125. Training solely at full length
   mismatches every short prompt. One length per batch, so examples stack without padding — the
   model has no key-padding mask, so left-padding would inject tokens it would wrongly attend to.

**Schedule: WSD** (warmup → stable → decay), not cosine. Cosine-to-zero would leave this branch
un-extendable without a re-warm spike — the exact tax phases 2 and 3 of this project paid. WSD lets
you stop after the stable phase and continue later for free.

Runs on CPU in ~5 minutes. It branches from `ckpt3` and writes `ckpt_autoreg.pt`; the pretrained
checkpoint is never modified.

In [62]:
import numpy as np, time, json as _json
from mini_enc_transformer import IGNORE_INDEX

torch.set_num_threads(4)          # leave headroom; a GPU run may be in progress
torch.manual_seed(0); np.random.seed(0)
rng = np.random.default_rng(0)

STEPS, BATCH, PEAK_LR, EOS_FRAC, MIN_CTX = 400, 8, 3e-5, 0.30, 8

# Replay draws from BOTH corpora, matching what phase 3 actually pretrained on.
# TinyStories alone would bias the fine-tune toward children's-story register.
CORPORA = ['tinystories', 'ultrafineweb_en']
SCANS, EOSPOS = [], []
for name in CORPORA:
    man = _json.load(open(f'data/{name}.manifest.json'))
    mm = np.memmap(f'data/{name}.bin', dtype=np.uint16, mode='r',
                   shape=(man['target_tokens'],))
    s = np.asarray(mm[:min(man['tokens_written'], 30_000_000)])
    e = np.flatnonzero(s == EOS_ID)
    SCANS.append(s); EOSPOS.append(e[(e > SEQ_LEN) & (e < len(s) - 1)])
    print(f'{name}: {len(s):,} tokens | {len(EOSPOS[-1]):,} document boundaries')


40,000,000 tokens scanned | 177,574 document boundaries


In [63]:
def make_batch(bs, ctx_len):
    """Left context + 3 masks; supervise ONLY the first mask slot with the true next token.

    One context length per batch, so rows stack without padding -- the model has no
    key-padding mask, so left-padding would inject tokens it would wrongly attend to.
    Context length is randomised because at inference the window GROWS: the first
    generated token sees only the prompt, later ones see the full 125.

    The corpus is chosen per example, so each batch mixes TinyStories and UltraFineWeb.
    """
    xs, ys = [], []
    for _ in range(bs):
        k = int(rng.integers(len(SCANS)))          # <- mix the replay corpora
        scan, eos_at = SCANS[k], EOSPOS[k]
        if rng.random() < EOS_FRAC and len(eos_at):
            tgt = int(rng.choice(eos_at))          # next token IS eos -> teaches stopping
        else:
            tgt = int(rng.integers(SEQ_LEN, len(scan) - 1))
        ctx = np.asarray(scan[tgt - ctx_len:tgt]).astype(np.int64)
        xs.append(np.concatenate([ctx, np.full(N_MASKS, MASK_ID, dtype=np.int64)]))
        lab = np.full(ctx_len + N_MASKS, IGNORE_INDEX, dtype=np.int64)
        lab[ctx_len] = int(scan[tgt])
        ys.append(lab)
    return torch.from_numpy(np.stack(xs)), torch.from_numpy(np.stack(ys))

def wsd(step, total, warmup_frac=0.10, decay_frac=0.30):
    """Warmup-Stable-Decay (MiniCPM 2024). The flat middle keeps the run extendable:
    cosine-to-zero would force a re-warm spike to continue, the tax phases 2 and 3 paid."""
    w, d = int(warmup_frac * total), int(decay_frac * total)
    if step < w:            return step / max(1, w)
    if step < total - d:    return 1.0
    return max(0.0, (total - step) / max(1, d))


In [ ]:
model.train()
opt = torch.optim.AdamW(model.parameters(), lr=PEAK_LR, weight_decay=0.0)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: wsd(s, STEPS))

t0, run = time.time(), 0.0
for step in range(1, STEPS + 1):
    L = int(np.random.randint(MIN_CTX, SEQ_LEN - N_MASKS + 1))   # randomised context length
    x, y = make_batch(BATCH, L)
    loss = model(x, y)['loss']
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step(); sched.step()
    run += loss.item()
    if step % 50 == 0:
        print(f'step {step}/{STEPS} loss {run/50:.4f} lr {sched.get_last_lr()[0]:.2e} '
              f'({(time.time()-t0)/step:.2f}s/step)'); run = 0.0
model.eval()
torch.save({'model': model.state_dict(), 'base': CKPT, 'steps': STEPS}, './checkpoints/ckpt_autoreg/ckpt_autoreg.pt')
print(f'done in {(time.time()-t0)/60:.1f} min -> ./checkpoints/ckpt_autoreg/ckpt_autoreg.pt')

fixed context and then autoreg doesn't help in avoiding loops.

In [ ]:
CKPT = './checkpoints/ckpt_autoreg/ckpt_autoreg_fixedctx.pt'

tk, ids = build_tokenizer("allenai/OLMo-1B-hf")
model = build_model(C(), ids)
model.load_state_dict(torch.load(CKPT, map_location="cpu")["model"])
model.eval()

print(f"params {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

params 28.7M


In [65]:
# Same prompts, same seed, after the fine-tune. Compare against the greedy output above.
for p in PROMPTS:
    print(f'> {p}\n  {generate(p, 30)}\n')

> Once upon a time there was a little girl who
  Once upon a time there was a little girl who loved to play with her friends.

> The movie was
  The movie was made by the two brothers and the two brothers and the two brothers and the two brothers and the two brothers and the two brothers and the two brothers and

> The capital of France is
  The capital of France is the capital of France.

> I watched this film last night and
  I watched this film last night and was a little bit bit bit bit bit bit bit bit bit bit bit bit bit bit bit bit bit bit bit bit bit bit bit bit bit bit bit

> I watched X-Men: Days. The name of the movie was 
  I watched X-Men: Days. The name of the movie was icky.

> Goku
  Goku, and the two two two two two two two two two two two two two two two two two two two two two two two two two two two



In [ ]:
CKPT = './checkpoints/ckpt_autoreg/ckpt_autoreg.pt' # with randomised context length, 400 steps, 8 batch, 3e-5 lr, 30% eos fraction

tk, ids = build_tokenizer("allenai/OLMo-1B-hf")
model = build_model(C(), ids)
model.load_state_dict(torch.load(CKPT, map_location="cpu")["model"])
model.eval()

print(f"params {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

params 28.7M


In [68]:
# Same prompts, same seed, after the fine-tune. Compare against the greedy output above.
for p in PROMPTS:
    print(f'> {p}\n  {generate(p, 30)}\n')

> Once upon a time there was a little girl who
  Once upon a time there was a little girl who loved to play with her friends.

> The movie was
  The movie was made on the TV.

> The capital of France is
  The capital of France is the capital of France.

> I watched this film last night and
  I watched this film last night and was so excited to see the film.

> I watched X-Men: Days. The name of the movie was 
  I watched X-Men: Days. The name of the movie was icky.

> Goku
  Goku.



---

# Fact retrieval from context: teaching an induction circuit

`"I watched Batman. The name of the movie was ___"` fails. This is a **copying**
(induction) failure, not a knowledge failure: the answer is right there in the context, and
getting it right only requires attending back to the earlier span and copying it forward.

Two reasons the model can't:

1. **Almost no machinery** — induction is usually implemented by a two-layer circuit
   (previous-token head feeding a copy head). With 4 layers x 4 heads there is very little
   room for one to form incidentally.
2. **Nothing rewarded it.** MLM masks tokens at random; a masked token whose answer appears
   verbatim earlier in the same window is rare, so the gradient signal for copying is thin.

Measured baseline on held-out entities: **1%**. The model effectively cannot copy.

## The fix

Synthetic templates where the answer is *provably* present earlier, so copying is the only
way to be right:

```
I watched Batman. The name of the movie was [MASK][MASK][MASK]
                                             ^ supervised = ' Batman'
```

Design choices that matter:

- **Held-out entities.** 200 entity tokens are excluded from training, so the evaluation
  measures *copying* rather than memorisation of particular names.
- **50/50 mix with the continuation objective.** Training purely on templates would collapse
  the model onto them -- the same forgetting lesson replay solved in phase 3.
- **One template per batch**, so rows share a sequence length (no key-padding mask available).
- **WSD** again, keeping the branch extendable.

In [ ]:
import re

# Entity pool: real single-token capitalised words, split train / held-out so the
# evaluation measures COPYING rather than memorising particular names.
ENTS = [t for t in range(EOS_ID)
        if (s := tk.decode([t])).startswith(' ') and s[1:].isalpha()
        and s[1:2].isupper() and 3 <= len(s[1:]) <= 12]
rng.shuffle(ENTS)
HELD, TRAIN_ENTS = ENTS[:200], ENTS[200:]
print(f'entity pool: {len(TRAIN_ENTS):,} train / {len(HELD)} held out')

# TWO entities, each referenced several times and interleaved. With a single entity the
# task collapses to "copy the one capitalised word" -- the model solved that instantly
# and then over-copied everywhere. Two entities force it to resolve WHICH one a slot
# refers to, which is the actual induction skill.
TEMPLATES = [
    'We went to{E} last year with{A}. I loved chatting with{E} while me and{A} walked around the city. I hope to visit{E} again soon.',
    '{E} and{A} were friends.{E} liked to run and{A} liked to swim. One day{E} asked{A} to play together.',
    'I watched{E} with{A}. The name of the movie was{E} and my friend was{A}. We both loved{E} a lot.',
    'The book{E} was written by{A}. Later{A} wrote a sequel to{E}, and everyone praised{A} for it.',
    'She met{E} at the park and{A} at the store. Later she called{E}, then she texted{A} about{E}.',
    'My dog{E} played with my cat{A}.{E} barked loudly while{A} slept. Then{E} ran to find{A}.',
    '{A} gave{E} a present.{E} thanked{A} and told{A} that{E} was very happy.',
    'The team{E} beat the team{A}. Fans of{E} cheered while fans of{A} left early, but{E} celebrated.',
]

SPLIT = re.compile(r'(\{[EA]\})')

def slots(tmpl):
    """Placeholder occurrences that have an EARLIER occurrence of the SAME entity --
    only those are answerable by copying."""
    parts = SPLIT.split(tmpl)
    out, seen = [], set()
    for i, part in enumerate(parts):
        if part in ('{E}', '{A}'):
            if part in seen:
                out.append(i)
            seen.add(part)
    return parts, out


entity pool: 7,109 train / 200 held out


In [70]:
def copy_batch(bs):
    """One (template, slot) per batch so rows share a length; single-token entities
    keep it exact. The answer is whichever entity that slot refers to -- recoverable
    only by attending back to its earlier occurrence."""
    tmpl = TEMPLATES[int(rng.integers(len(TEMPLATES)))]
    parts, cand = slots(tmpl)
    target_i = int(rng.choice(cand))
    xs, ys = [], []
    for _ in range(bs):
        e, a_ = (int(x) for x in rng.choice(TRAIN_ENTS, size=2, replace=False))
        sub = {'{E}': tk.decode([e]), '{A}': tk.decode([a_])}
        prefix = ''.join(sub.get(p, p) for p in parts[:target_i])
        answer = e if parts[target_i] == '{E}' else a_
        ctx = tk(prefix, add_special_tokens=False)['input_ids']
        xs.append(np.array(ctx + [MASK_ID] * N_MASKS, dtype=np.int64))
        lab = np.full(len(ctx) + N_MASKS, IGNORE_INDEX, dtype=np.int64)
        lab[len(ctx)] = answer
        ys.append(lab)
    n = min(len(x) for x in xs)
    return (torch.from_numpy(np.stack([x[-n:] for x in xs])),
            torch.from_numpy(np.stack([y[-n:] for y in ys])))

@torch.no_grad()
def copy_acc(templates, n=120):
    """Accuracy on entities never trained on."""
    model.eval(); ok = 0
    for i in range(n):
        tmpl = templates[i % len(templates)]
        parts, cand = slots(tmpl)
        target_i = cand[i % len(cand)]
        e, a_ = int(HELD[i % len(HELD)]), int(HELD[(i + 37) % len(HELD)])
        if e == a_: continue
        sub = {'{E}': tk.decode([e]), '{A}': tk.decode([a_])}
        prefix = ''.join(sub.get(p, p) for p in parts[:target_i])
        answer = e if parts[target_i] == '{E}' else a_
        ctx = tk(prefix, add_special_tokens=False)['input_ids']
        lg = model(torch.tensor([ctx + [MASK_ID] * N_MASKS]))['logits'][0, len(ctx)]
        ok += int(int(lg.argmax()) == answer)
    model.train(); return ok / n

print(f'BEFORE  trained-style {copy_acc(TEMPLATES):.3f}')


held-out copy accuracy BEFORE: 0.010


In [ ]:
STEPS_IND, COPY_FRAC = 600, 0.25   # 0.5 made it over-copy ('I read Dracula and Dracula')

model.train()
opt = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.0)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: wsd(s, STEPS_IND))

t0, run = time.time(), 0.0
for step in range(1, STEPS_IND + 1):
    if rng.random() < COPY_FRAC:
        x, y = copy_batch(BATCH)                                              # induction
    else:
        x, y = make_batch(BATCH, int(rng.integers(MIN_CTX, SEQ_LEN - N_MASKS + 1)))  # replay
    loss = model(x, y)['loss']
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step(); sched.step()
    run += loss.item()
    if step % 100 == 0:
        print(f'step {step}/{STEPS_IND} loss {run/100:.4f} lr {sched.get_last_lr()[0]:.2e}'); run = 0.0

model.eval()
torch.save({'model': model.state_dict(), 'base': CKPT}, './checkpoints/ckpt_autoreg/ckpt_induction.pt')
print(f'done in {(time.time()-t0)/60:.1f} min')


In [77]:
model.load_state_dict(torch.load('./checkpoints/ckpt_autoreg/ckpt_induction.pt', map_location="cpu")["model"])

<All keys matched successfully>

In [90]:
# The actual failing prompts, plus entities the model never trained on.
for p in ['I watched Batman. The name of the movie was',
          'My friend Sarah came over. I said hello to',
          'The book was called Dracula. I read',
          'I watched Bleach: Thousand Year Blood War. Osman really liked it. The name of the movie was']:
    print(f'> {p}\n  {generate(p, max_gen=8, temperature=0.3)}\n')

> I watched Batman. The name of the movie was
  I watched Batman. The name of the movie was Batman.

> My friend Sarah came over. I said hello to
  My friend Sarah came over. I said hello to Sarah.

> The book was called Dracula. I read
  The book was called Dracula. I read Dracula.

> I watched Bleach: Thousand Year Blood War. Osman really liked it. The name of the movie was
  I watched Bleach: Thousand Year Blood War. Osman really liked it. The name of the movie was Osman's Blood War.



In [91]:
# Is it a real copy operation, or did it memorise the TRAINED templates? Test on
# phrasings never seen in training -- including one where an entity starts the text.
UNSEEN_TEMPLATES = [
    'The film{E} was great and{A} agreed. Everyone praised{E} but nobody praised{A}, so{E}',
    '{E} sat next to{A} on the bus.{A} smiled at{E} and then{A}',
    'Yesterday I met{E} and{A}. I called{E} first, and after that I called{A}, because{E}',
]

print(f'TRAINED templates, held-out entities : {copy_acc(TEMPLATES):.3f}')
print(f'UNSEEN  templates, held-out entities : {copy_acc(UNSEEN_TEMPLATES):.3f}')


TRAINED templates, held-out entities : 0.892
UNSEEN  templates, held-out entities : 0.000


### Result, and the regression it introduced

`1% -> 100%` on trained templates and **98.3% on unseen phrasings** (including the entity at
position 0) — so this is a genuine copy operation, not a positional shortcut.

**But it over-copies.** With `COPY_FRAC = 0.5` the model became too eager:

```
My friend Sarah came over. I said hello to Sarah came over and said hello to Sarah
The book was called Dracula. I read Dracula and Dracula.
```

It learned *to* copy but not *when to stop*. Story prompts survived intact, so the replay half
protected general fluency — just not proportionality. Fixes, cheapest first:

- lower `COPY_FRAC` to ~0.25
- add negative examples whose continuation is deliberately **not** a copy, so the model learns
  when copying is inappropriate

This is the recurring lesson of the project in miniature: a targeted objective fixes the thing
it targets and distorts what sits next to it, and only replay keeps the distortion bounded.